# Conv-LoRA Backdoor Fine-Tuning

In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'defense').exists() and (candidate / 'record').exists() and (candidate / 'notebooks_finetuning').exists():
            return candidate
    raise FileNotFoundError('Could not locate the cloned backdoor_finetuning repo. Start Jupyter from the repo or its notebooks_finetuning folder.')


REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_finetuning'
LORA_DIR = NOTEBOOK_DIR / 'convlora'
ZIP_SEARCH_DIRS = [LORA_DIR, NOTEBOOK_DIR]

RUN_SOURCE = 'cifar10_preactresnet18_bpp_0_1.zip'


def resolve_run_source(source):
    source = Path(source).expanduser()
    if source.is_absolute():
        return source
    for folder in ZIP_SEARCH_DIRS:
        candidate = folder / source
        if candidate.exists():
            return candidate
    return LORA_DIR / source


RUN_SOURCE = resolve_run_source(RUN_SOURCE)
EXTRACT_ROOT = LORA_DIR / 'extracted'
OUTPUT_ROOT = LORA_DIR / 'outputs'
DATA_ROOT = LORA_DIR / 'data'

# Experiment settings.
SEED = 0
EPOCHS = 20
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
NUM_WORKERS = 0
LR = 1e-3
WEIGHT_DECAY = 1e-4
CLEAN_TRAIN_LIMIT = None       
CLEAN_TEST_LIMIT = None
BD_TEST_LIMIT = 2000          
DOWNLOAD_CIFAR10 = True
TARGET_LABEL_OVERRIDE = None   

# Conv-LoRA settings
LORA_RANK = 4
LORA_ALPHA = 8
UNFREEZE_CLASSIFIER = True

# BackdoorBench CIFAR-10 normalization
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.247, 0.243, 0.261)

print('Run source:', RUN_SOURCE)
print('LoRA workspace:', LORA_DIR)


In [ ]:
import importlib.util
import json
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

missing = []
for module_name, package_name in [
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('pandas', 'pandas'),
    ('matplotlib', 'matplotlib'),
    ('tqdm', 'tqdm'),
    ('PIL', 'pillow'),
    ('numpy', 'numpy'),
]:
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)

if missing:
    raise ImportError(
        'Missing packages: ' + ', '.join(missing) +
        '. Install them, then restart this kernel.'
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm


def pick_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


DEVICE = pick_device()
LORA_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == 'mps' and hasattr(torch, 'mps') and hasattr(torch.mps, 'manual_seed'):
    torch.mps.manual_seed(SEED)

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type == 'mps':
    print('Using Apple Metal Performance Shaders. PYTORCH_ENABLE_MPS_FALLBACK=1 is set for unsupported ops.')


In [ ]:
def extract_run(source, extract_root, force=False):
    source = Path(source).expanduser()
    if not source.exists():
        raise FileNotFoundError(source)

    if source.is_dir():
        return source

    if source.suffix.lower() != '.zip':
        raise ValueError(f'RUN_SOURCE must be a folder or .zip file, got {source}')

    run_dir = extract_root / source.stem
    marker = run_dir / '.extract_complete'
    if force and run_dir.exists():
        shutil.rmtree(run_dir)
    if marker.exists():
        return run_dir

    tmp_dir = extract_root / f'.{source.stem}_tmp'
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)

    with zipfile.ZipFile(source) as zf:
        zf.extractall(tmp_dir)

    entries = [p for p in tmp_dir.iterdir() if p.name != '__MACOSX']
    if len(entries) == 1 and entries[0].is_dir():
        extracted_root = entries[0]
    else:
        extracted_root = tmp_dir

    if run_dir.exists():
        shutil.rmtree(run_dir)
    if extracted_root == tmp_dir:
        tmp_dir.rename(run_dir)
    else:
        shutil.move(str(extracted_root), str(run_dir))
        shutil.rmtree(tmp_dir, ignore_errors=True)

    marker.write_text(time.strftime('%Y-%m-%d %H:%M:%S'))
    return run_dir


run_dir = extract_run(RUN_SOURCE, EXTRACT_ROOT)
RUN_NAME = run_dir.stem
attack_result_path = run_dir / 'attack_result.pt'
bd_test_dir = run_dir / 'bd_test_dataset'
bd_train_dir = run_dir / 'bd_train_dataset'

if not attack_result_path.exists():
    raise FileNotFoundError(f'Missing attack_result.pt in {run_dir}')
if not bd_test_dir.exists():
    raise FileNotFoundError(f'Missing bd_test_dataset in {run_dir}')

print('Run name:', RUN_NAME)
print('Run directory:', run_dir)
print('attack_result:', attack_result_path)
print('bd_test_dataset:', bd_test_dir)
print('bd_train_dataset exists:', bd_train_dir.exists())


In [ ]:
# Preactivation ResNet18 definition used by the BackdoorBench checkpoints
class PreActBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.ind = None
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False)
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        if self.ind is not None:
            out += shortcut[:, self.ind, :, :]
        else:
            out += shortcut
        return out


class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.linear = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.linear(out)


def PreActResNet18(num_classes=10):
    return PreActResNet(PreActBlock, [2, 2, 2, 2], num_classes=num_classes)


def make_model(model_name, num_classes):
    model_name = str(model_name).lower()
    if model_name != 'preactresnet18':
        raise ValueError(
            f'This standalone notebook only supports preactresnet18. Found {model_name!r}. '
            'Use the repo-based notebook for other architectures.'
        )
    return PreActResNet18(num_classes=num_classes)


In [ ]:
class LoRAConv2d(nn.Module):
    def __init__(self, conv, rank=4, alpha=8):
        super().__init__()
        if conv.groups != 1:
            raise ValueError('LoRAConv2d only supports ungrouped convolutions.')
        self.conv = conv
        self.scale = alpha / rank
        for param in self.conv.parameters():
            param.requires_grad = False

        self.lora_a = nn.Conv2d(
            conv.in_channels,
            rank,
            conv.kernel_size,
            conv.stride,
            conv.padding,
            conv.dilation,
            padding_mode=conv.padding_mode,
            bias=False,
        )
        self.lora_b = nn.Conv2d(rank, conv.out_channels, kernel_size=1, bias=False)
        nn.init.kaiming_uniform_(self.lora_a.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_b.weight)

    def forward(self, x):
        return self.conv(x) + self.scale * self.lora_b(self.lora_a(x))


def add_conv_lora(module, rank=4, alpha=8):
    replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Conv2d):
            setattr(module, name, LoRAConv2d(child, rank=rank, alpha=alpha))
            replaced += 1
        else:
            replaced += add_conv_lora(child, rank=rank, alpha=alpha)
    return replaced


def mark_lora_trainable(model, unfreeze_classifier=True):
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        lname = name.lower()
        if 'lora_' in lname:
            param.requires_grad = True
        elif unfreeze_classifier and ('linear' in lname or 'fc' in lname or 'classifier' in lname):
            param.requires_grad = True


def torch_load_compat(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def strip_module_prefix(state_dict):
    return {key[7:] if key.startswith('module.') else key: value for key, value in state_dict.items()}


def looks_like_state_dict(candidate):
    if not isinstance(candidate, dict) or not candidate:
        return False
    values = list(candidate.values())
    tensor_count = sum(torch.is_tensor(value) for value in values)
    return tensor_count > 0 and tensor_count == len(values)


def extract_state_dict(obj):
    if isinstance(obj, dict):
        if looks_like_state_dict(obj):
            return obj
        for key in ('model', 'model_state_dict', 'state_dict', 'net', 'network'):
            if key in obj:
                return extract_state_dict(obj[key])
    raise ValueError('Could not find a model state_dict in the checkpoint.')


def load_state_dict_flexible(model, state_dict):
    state_dict = strip_module_prefix(state_dict)
    try:
        model.load_state_dict(state_dict, strict=True)
        return 'strict'
    except RuntimeError as strict_error:
        model_state = model.state_dict()
        model_keys = list(model_state.keys())
        state_keys = list(state_dict.keys())
        if len(model_keys) == len(state_keys):
            remapped = {}
            compatible = True
            for model_key, state_key in zip(model_keys, state_keys):
                if tuple(model_state[model_key].shape) != tuple(state_dict[state_key].shape):
                    compatible = False
                    break
                remapped[model_key] = state_dict[state_key]
            if compatible:
                print('WARNING: strict load failed; loaded checkpoint by positional key remapping.')
                model.load_state_dict(remapped, strict=True)
                return 'remapped_by_order'
        raise strict_error


def count_parameters(model):
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    total = sum(param.numel() for param in model.parameters())
    return trainable, total


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomCrop((32, 32), padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])


class IntFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform
        self.samples = []
        if not self.root.exists():
            raise FileNotFoundError(self.root)

        class_dirs = [p for p in self.root.iterdir() if p.is_dir()]
        class_dirs = sorted(class_dirs, key=lambda p: int(p.name) if p.name.isdigit() else p.name)
        if class_dirs:
            for class_dir in class_dirs:
                try:
                    label = int(class_dir.name)
                except ValueError as exc:
                    raise ValueError(f'Expected numeric class folder, got {class_dir.name}') from exc
                for image_path in sorted(class_dir.rglob('*')):
                    if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                        self.samples.append((image_path, label))
        else:
            for image_path in sorted(self.root.rglob('*')):
                if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    self.samples.append((image_path, -1))

        if not self.samples:
            raise ValueError(f'No images found under {self.root}')
        self.labels = [label for _, label in self.samples]
        self.classes = sorted(label for label in set(self.labels) if label >= 0)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]
        image = Image.open(image_path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def maybe_limit_dataset(dataset, limit, seed=0):
    if limit is None or limit >= len(dataset):
        return dataset
    generator = torch.Generator().manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=generator)[:limit].tolist()
    return Subset(dataset, indices)


def nested_get(obj, name):
    if isinstance(obj, dict):
        return obj.get(name)
    return getattr(obj, name, None)


def coerce_int(value):
    if value is None:
        return None
    if hasattr(value, 'item'):
        value = value.item()
    if isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def infer_target_label(raw_attack, bd_dataset, num_classes):
    if TARGET_LABEL_OVERRIDE is not None:
        return int(TARGET_LABEL_OVERRIDE), 'override'

    if isinstance(raw_attack, dict):
        for key in ('target_label', 'attack_target', 'target_class', 'poison_label', 'target'):
            value = coerce_int(raw_attack.get(key))
            if value is not None:
                return value, f'attack_result[{key!r}]'

        args_obj = raw_attack.get('args')
        for key in ('target_label', 'attack_target', 'target_class', 'poison_label', 'target'):
            value = coerce_int(nested_get(args_obj, key))
            if value is not None:
                return value, f'attack_result.args.{key}'

    if getattr(bd_dataset, 'classes', None):
        missing = sorted(set(range(num_classes)) - set(bd_dataset.classes))
        if len(missing) == 1:
            return missing[0], 'missing class folder in bd_test_dataset'

    return 0, 'fallback default'


bd_test_dataset = IntFolderDataset(bd_test_dir, transform=test_transform)
clean_train_dataset = CIFAR10(root=str(DATA_ROOT), train=True, download=DOWNLOAD_CIFAR10, transform=train_transform)
clean_test_dataset = CIFAR10(root=str(DATA_ROOT), train=False, download=DOWNLOAD_CIFAR10, transform=test_transform)

clean_train_dataset = maybe_limit_dataset(clean_train_dataset, CLEAN_TRAIN_LIMIT, seed=SEED)
clean_test_dataset = maybe_limit_dataset(clean_test_dataset, CLEAN_TEST_LIMIT, seed=SEED)
bd_test_dataset = maybe_limit_dataset(bd_test_dataset, BD_TEST_LIMIT, seed=SEED)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
}
train_loader = DataLoader(clean_train_dataset, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs)
clean_test_loader = DataLoader(clean_test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, **loader_kwargs)
bd_test_loader = DataLoader(bd_test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, **loader_kwargs)

raw_attack = torch_load_compat(attack_result_path, map_location='cpu')
model_name = raw_attack.get('model_name', 'preactresnet18') if isinstance(raw_attack, dict) else 'preactresnet18'
num_classes = int(raw_attack.get('num_classes', 10)) if isinstance(raw_attack, dict) else 10
full_bd_for_target = IntFolderDataset(bd_test_dir, transform=None)
target_label, target_source = infer_target_label(raw_attack, full_bd_for_target, num_classes)

print('Clean train samples:', len(clean_train_dataset))
print('Clean test samples:', len(clean_test_dataset))
print('BD test samples:', len(bd_test_dataset))
print('BD test labels:', Counter(full_bd_for_target.labels))
print('Model:', model_name)
print('Num classes:', num_classes)
print('Target label:', target_label, f'({target_source})')


In [ ]:
base_model = make_model(model_name, num_classes=num_classes)
state_dict = extract_state_dict(raw_attack)
load_mode = load_state_dict_flexible(base_model, state_dict)

wrapped_convs = add_conv_lora(base_model, rank=LORA_RANK, alpha=LORA_ALPHA)
mark_lora_trainable(base_model, unfreeze_classifier=UNFREEZE_CLASSIFIER)
model = base_model.to(DEVICE)

trainable, total = count_parameters(model)
print('Checkpoint load mode:', load_mode)
print('Conv layers wrapped with LoRA:', wrapped_convs)
print(f'Trainable params: {trainable:,}/{total:,} ({100 * trainable / total:.2f}%)')


In [ ]:
def to_device(batch):
    x, y = batch[0], batch[1]
    non_blocking = DEVICE.type == 'cuda'
    return x.to(DEVICE, non_blocking=non_blocking), y.to(DEVICE, non_blocking=non_blocking)


@torch.no_grad()
def evaluate(loader, target_label=None):
    model.eval()
    correct = 0
    total = 0
    for batch in loader:
        x, y = to_device(batch)
        pred = model(x).argmax(dim=1)
        if target_label is None:
            correct += (pred == y).sum().item()
        else:
            correct += (pred == int(target_label)).sum().item()
        total += y.numel()
    if total == 0:
        raise RuntimeError('Empty evaluation loader')
    return correct / total


before_clean_acc = evaluate(clean_test_loader)
before_asr = evaluate(bd_test_loader, target_label=target_label)
print(f'Before Conv-LoRA | Clean ACC: {before_clean_acc:.4f} | ASR: {before_asr:.4f}')


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    [param for param in model.parameters() if param.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

history = [{
    'epoch': 0,
    'phase': 'before_lora',
    'run_name': RUN_NAME,
    'architecture': str(model_name).lower(),
    'target_label': target_label,
    'train_acc': None,
    'clean_acc': before_clean_acc,
    'asr': before_asr,
    'loss': None,
    'asr_reduction_from_before': 0.0,
}]

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(train_loader, desc=f'Epoch {epoch:03d}/{EPOCHS}', leave=False)
    for batch in progress:
        x, y = to_device(batch)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = y.numel()
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += batch_size
        progress.set_postfix(loss=f'{loss.item():.4f}')

    train_acc = correct / total
    avg_loss = total_loss / total
    clean_acc = evaluate(clean_test_loader)
    asr = evaluate(bd_test_loader, target_label=target_label)

    history.append({
        'epoch': epoch,
        'phase': 'lora_finetune',
        'run_name': RUN_NAME,
        'architecture': str(model_name).lower(),
        'target_label': target_label,
        'train_acc': train_acc,
        'clean_acc': clean_acc,
        'asr': asr,
        'loss': avg_loss,
        'asr_reduction_from_before': before_asr - asr,
    })

    print(
        f'Epoch {epoch:03d}/{EPOCHS} | '
        f'Train ACC {train_acc:.4f} | Clean ACC {clean_acc:.4f} | '
        f'ASR {asr:.4f} | Loss {avg_loss:.4f}'
    )

    if DEVICE.type == 'mps' and hasattr(torch, 'mps'):
        torch.mps.empty_cache()

final_clean_acc = evaluate(clean_test_loader)
final_asr = evaluate(bd_test_loader, target_label=target_label)
history.append({
    'epoch': EPOCHS,
    'phase': 'final',
    'run_name': RUN_NAME,
    'architecture': str(model_name).lower(),
    'target_label': target_label,
    'train_acc': None,
    'clean_acc': final_clean_acc,
    'asr': final_asr,
    'loss': None,
    'asr_reduction_from_before': before_asr - final_asr,
})

history_df = pd.DataFrame(history)
print('\nFinal Conv-LoRA result')
print(f'Before: ACC={before_clean_acc:.4f}, ASR={before_asr:.4f}')
print(f'After:  ACC={final_clean_acc:.4f}, ASR={final_asr:.4f}')
print(f'ASR reduction: {before_asr - final_asr:.4f}')
display(history_df)


In [ ]:
output_dir = OUTPUT_ROOT / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)

metrics_path = output_dir / 'conv_lora_metrics.csv'
checkpoint_path = output_dir / 'conv_lora_model.pt'
summary_path = output_dir / 'conv_lora_summary.json'
plot_path = output_dir / 'conv_lora_curves.png'

history_df.to_csv(metrics_path, index=False)

torch.save({
    'model_state_dict': model.state_dict(),
    'run_name': RUN_NAME,
    'source': str(RUN_SOURCE),
    'architecture': str(model_name).lower(),
    'num_classes': num_classes,
    'target_label': target_label,
    'target_label_source': target_source,
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'unfreeze_classifier': UNFREEZE_CLASSIFIER,
    'normalization_mean': CIFAR10_MEAN,
    'normalization_std': CIFAR10_STD,
    'history': history,
}, checkpoint_path)

summary = {
    'run_name': RUN_NAME,
    'source': str(RUN_SOURCE),
    'device': str(DEVICE),
    'architecture': str(model_name).lower(),
    'num_classes': num_classes,
    'target_label': target_label,
    'target_label_source': target_source,
    'epochs': EPOCHS,
    'before_clean_acc': before_clean_acc,
    'before_asr': before_asr,
    'final_clean_acc': final_clean_acc,
    'final_asr': final_asr,
    'asr_reduction': before_asr - final_asr,
}
summary_path.write_text(json.dumps(summary, indent=2))

ax = history_df.plot(x='epoch', y=['clean_acc', 'asr'], marker='o', figsize=(8, 4))
ax.set_ylim(0, 1)
ax.set_ylabel('rate')
ax.grid(True, alpha=0.3)
ax.figure.tight_layout()
ax.figure.savefig(plot_path, dpi=160)
plt.show()

print('Saved files:')
print(metrics_path)
print(checkpoint_path)
print(summary_path)
print(plot_path)
